In [10]:
import polars as pl
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import concurrent.futures
from tqdm import tqdm
import time
import os
import math

In [12]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
csv_reviews_output_scores = '../results/vader/yelp_academic_dataset_review_scored.csv'
batch_size = 200000
total_rows = 6_990_280

def process_chunk(texts: list[str]):
    '''
    Inicializa un SentimentIntensityAnalyzer y devuelve el "score" de varios textos

    Args:
        texts (list[str]): Lista que contiene varios textos de reseñas
    '''
    
    sentimentAnalyzer = SentimentIntensityAnalyzer()
    return [sentimentAnalyzer.polarity_scores(str(t))['compound'] for t in texts]

def analyze_sentiment_vader(input_csv: str, output_csv: str):
    '''
    Realiza el análisis de sentimiento sobre un csv que contiene reseñas con texto
    y devuelve un csv que añade un "score" a cada reseña según el lexicon VADER

    Args:
        input_csv (str): Ruta del csv que contiene las reseñas y dentro de cada reseña un texto
        output_csv (str): Ruta del csv al que se escribirá las reseñas junto con su "sentiment score" según el lexicon VADER
    '''
    
    try:
        nltk.data.find('sentiment/vader_lexicon.zip')
    except LookupError:
        nltk.download('vader_lexicon')
        
    num_cores = os.cpu_count()
    print(f"Núcleos CPU detectados: {num_cores}")

    start_time = time.time()

    try:
        temp_df = pl.read_csv(input_csv, n_rows=1, ignore_errors=True)
        columns = temp_df.columns + ['vader_score']

        with open(output_csv, 'w') as f:
            f.write(",".join([f'"{c}"' for c in columns]) + "\n")
    except Exception as e:
        print(f"Error leyendo cabeceras: {e}")
        return

    reader = pl.read_csv_batched(input_csv, batch_size=batch_size, ignore_errors=True)

    with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
        with tqdm(total=total_rows, unit="reviews", desc="Progreso") as pbar:
            while True:
                batches = reader.next_batches(1)

                if not batches:
                    break

                df_batch = batches[0]
                current_rows = df_batch.height

                texts = df_batch["text"].to_list()

                chunk_size = math.ceil(len(texts) / num_cores)
                chunks = [texts[i:i + chunk_size] for i in range(0, len(texts), chunk_size)]

                results_generator = executor.map(process_chunk, chunks)

                flat_scores = []
                for res in results_generator:
                    flat_scores.extend(res)
                
                df_scored = df_batch.with_columns(
                    pl.Series(name="vader_score", values=flat_scores, dtype=pl.Float64)
                )
                
                with open(output_csv, "a") as f:
                    df_scored.write_csv(f, include_header=False)
                
                pbar.update(current_rows)
                
    end_time = time.time()

    print(f"Tiempo total: {(end_time - start_time) / 60:.2f} minutos.")

analyze_sentiment_vader(csv_reviews, csv_reviews_output_scores)

Núcleos CPU detectados: 12


Progreso: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6990280/6990280 [12:51<00:00, 9064.71reviews/s]

Tiempo total: 12.86 minutos.
